In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

In [2]:
# 1. Resolução dos caminhos dinâmicos (Raiz do projeto e Data lake simulado)
project_root = Path(".").resolve().parent.parent
mock_storage_dir = project_root / "data" / "gcp_storage_mock" / "aposta_ganha_raw_data_lake" / "raw"
db_path = project_root / "aposta_ganha_dw.db"

# 2. Leitura dos dados da camada Raw (Arquivos CSV)
print("Lendo os dados do Data Lake simulado...")
df_usuarios = pd.read_csv(mock_storage_dir / "dim_usuarios.csv")
df_transacoes = pd.read_csv(mock_storage_dir / "fato_transacoes.csv")

# 3. Conexão com o Data Warehouse (SQLite)
conn = sqlite3.connect(str(db_path))
cursor = conn.cursor()

print("Estruturado o Data Warehouse, Modelagem Star Schema...")

# 4. Limpeza prévia (drop)
cursor.execute('DROP TABLE IF EXISTS fato_transacoes')
cursor.execute('DROP TABLE IF EXISTS dim_usuarios')

# 5. DDL: Criação da tabela dimensão de usuários
# Definindo tipagem forte e a chave primária (user_id)
cursor.execute('''
CREATE TABLE dim_usuarios (
    user_id INTEGER PRIMARY KEY,
    nome TEXT,
    idade INTEGER,
    estado TEXT,
    data_cadastro DATE,
    preferencia_jogo TEXT,
    flag_vip INTEGER)''')

# 6. DDL: Criação da tabela de transações
# Difinindo tipagem forte e a relacionamento via chave estrangeira (foreign key)
cursor.execute('''
CREATE TABLE fato_transacoes (
    transacao_id TEXT PRIMARY KEY,
    user_id INTEGER,
    data_hora DATETIME,
    tipo_transacao TEXT,
    valor_transacao REAL,
    valor_ganho REAL,
    valor_perdido REAL,
    FOREIGN KEY (user_id) REFERENCES dim_usuarios (user_id))''')

# 7. Carregando os dados do pandas para as tabelas estruturadas
print("Inserindo os registros no Data Warehouse...")
df_usuarios.to_sql("dim_usuarios", conn, if_exists="append", index=False)
df_transacoes.to_sql("fato_transacoes", conn, if_exists="append", index=False)

# Confirma as transações e fecha a conexão
conn.commit()
conn.close()

print("O banco de dados 'aposta_ganha_dw' foi criado e o modelo Star Schema está 100% modelado.")

Lendo os dados do Data Lake simulado...
Estruturado o Data Warehouse, Modelagem Star Schema...
Inserindo os registros no Data Warehouse...
O banco de dados 'aposta_ganha_dw' foi criado e o modelo Star Schema está 100% modelado.
